<a href="https://colab.research.google.com/github/AlanChi0720/bio_ai/blob/main/B1_enzyme_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Track B — Notebook 1: Supervised Enzyme Classification on ESM-2 Embeddings

**Big idea:** In Phase 2 you saw that ESM-2 embeddings cluster proteins by family *without* any labels (unsupervised). Now we add labels and train a real classifier. This is the workhorse pattern in protein ML: **frozen pretrained model → small head trained on your task**.

**Task:** Predict enzyme commission (EC) class — the top-level number that classifies what an enzyme does:

| EC class | What it does |
|---|---|
| 1 | Oxidoreductases (redox reactions) |
| 2 | Transferases (move functional groups) |
| 3 | Hydrolases (cleave bonds with water) |
| 4 | Lyases (break bonds without water/oxidation) |

**ML approach:** ESM-2 (frozen) → mean-pooled embedding → logistic regression and MLP classifiers.

**Estimated time:** ~3 hours, GPU recommended.

In [ ]:
!pip install -q transformers torch

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import EsmTokenizer, EsmModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
sns.set_theme(style='whitegrid')

## 1. Download Sequences from UniProt

UniProt has a REST API that lets you search by EC class. We'll grab ~150 reviewed (SwissProt) sequences from each of 4 EC classes.

In [ ]:
def fetch_uniprot_by_ec(ec_top, n=150, max_len=400):
    import urllib.parse

    query = f'ec:{ec_top}.* AND reviewed:true AND length:[50 TO {max_len}]'
    params = urllib.parse.urlencode({
        'query': query,
        'format': 'tsv',
        'fields': 'accession,sequence,ec',
        'size': n
    })
    url = f'https://rest.uniprot.org/uniprotkb/search?{params}'

    with urllib.request.urlopen(url) as resp:
        data = resp.read().decode()
    df = pd.read_csv(io.StringIO(data), sep='\t')
    df = df.dropna(subset=['Sequence'])
    df['ec_top'] = ec_top
    return df

In [ ]:
frames = [fetch_uniprot_by_ec(c) for c in [1, 2, 3, 4]]
df = pd.concat(frames, ignore_index=True)
df = df.drop_duplicates(subset=['Sequence']).reset_index(drop=True)
print(df['ec_top'].value_counts())
df.head()

## 2. Load ESM-2 (Small) and Define Embedding Function

Same model as Phase 2 (`esm2_t6_8M_UR50D`). For real research you'd use a larger one (650M or 3B params), but the small one is fast enough to embed several hundred proteins on a free Colab GPU in a few minutes.

In [ ]:
model_name = 'facebook/esm2_t6_8M_UR50D'
tokenizer = EsmTokenizer.from_pretrained(model_name)
esm = EsmModel.from_pretrained(model_name).to(device)
esm.eval()  # frozen — we never train ESM-2 itself in this notebook
for p in esm.parameters():
    p.requires_grad = False
print('ESM-2 loaded and frozen.')

In [ ]:
@torch.no_grad()
def embed_batch(sequences, batch_size=8, max_len=400):
    """Mean-pooled ESM-2 embeddings (one vector per sequence)."""
    out = []
    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i+batch_size]
        toks = tokenizer(batch, return_tensors='pt', padding=True,
                         truncation=True, max_length=max_len).to(device)
        h = esm(**toks).last_hidden_state                # (B, L, D)
        mask = toks['attention_mask'].unsqueeze(-1).float()
        # Mean over real residues only (mask out padding)
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        out.append(pooled.cpu().numpy())
    return np.concatenate(out, axis=0)

In [ ]:
# Embed all sequences. On a free Colab T4 this takes ~2-5 min.
import time
t0 = time.time()
X = embed_batch(df['Sequence'].tolist())
y = df['ec_top'].values - 1   # convert to 0-indexed labels
print(f'Embedded {X.shape[0]} sequences in {time.time()-t0:.1f}s')
print(f'Embedding shape: {X.shape}  (samples, embedding_dim)')

## 3. Sanity Check — PCA First (Like Phase 2)

Before training a classifier, look at the embeddings. Do the four EC classes already separate visually?

In [ ]:
from sklearn.decomposition import PCA
coords = PCA(n_components=2).fit_transform(X)

fig, ax = plt.subplots(figsize=(7, 5))
for cls in range(4):
    m = (y == cls)
    ax.scatter(coords[m, 0], coords[m, 1], label=f'EC {cls+1}', alpha=0.6, s=30)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('ESM-2 embeddings of enzymes (PCA)')
ax.legend(); plt.tight_layout(); plt.show()

# Q: Compared to Phase 2 (kinases vs opsins), do you see cleaner or messier separation?
# 4 broad EC classes are noisier than 2 specific protein families — that's expected.

## 4. Train/Test Split + Logistic Regression Baseline

First baseline: a plain old logistic regression on the 320-dim ESM-2 embeddings. This is the standard "linear probe" used to evaluate pretrained representations.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

lr = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial')
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print(f'Logistic Regression accuracy: {accuracy_score(y_test, y_pred_lr):.3f}\n')
print(classification_report(y_test, y_pred_lr,
                            target_names=['EC1', 'EC2', 'EC3', 'EC4']))

## 5. MLP Head with PyTorch

Now train a small MLP on the same embeddings. Same training-loop pattern as the prerequisite notebook.

In [ ]:
class HeadMLP(nn.Module):
    def __init__(self, in_dim, n_classes=4, hidden=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_classes),
        )
    def forward(self, x):
        return self.net(x)

Xtr = torch.from_numpy(X_train).float().to(device)
ytr = torch.from_numpy(y_train).long().to(device)
Xte = torch.from_numpy(X_test).float().to(device)
yte = torch.from_numpy(y_test).long().to(device)

head = HeadMLP(in_dim=X.shape[1]).to(device)
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

history = []
for epoch in range(220):
    head.train(); opt.zero_grad()
    loss = loss_fn(head(Xtr), ytr)
    loss.backward(); opt.step()

    head.eval()
    with torch.no_grad():
        acc = (head(Xte).argmax(1) == yte).float().mean().item()
    history.append((loss.item(), acc))
    if epoch % 20 == 0:
        print(f'epoch {epoch:3d}  loss {loss.item():.3f}  test_acc {acc:.3f}')

with torch.no_grad():
    y_pred_mlp = head(Xte).argmax(1).cpu().numpy()
print(f'\nMLP final test accuracy: {accuracy_score(y_test, y_pred_mlp):.3f}')

In [ ]:
loss_curve = [h[0] for h in history]
acc_curve  = [h[1] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(loss_curve); axes[0].set_title('Train loss'); axes[0].set_xlabel('epoch')
axes[1].plot(acc_curve);  axes[1].set_title('Test accuracy'); axes[1].set_xlabel('epoch')
plt.tight_layout(); plt.show()

## 6. Confusion Matrix — Which EC Classes Get Mixed Up?

Accuracy alone hides which mistakes the model makes. The confusion matrix shows you, for each true class, which class the model predicted. This often reveals real biological similarity.

In [ ]:
labels_str = ['EC1 Oxidoreductase', 'EC2 Transferase', 'EC3 Hydrolase', 'EC4 Lyase']
cm = confusion_matrix(y_test, y_pred_mlp, normalize='true')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels_str, yticklabels=labels_str, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix (normalized by true class)')
plt.tight_layout(); plt.show()

# Q: Which two classes get confused most? Does that make biological sense?
# i think is ec2 and ec4, i think it does, for ec2 is transferase, it's between 2 or multiple protein, so its hrad to predict, i guess?
# (E.g., transferases and hydrolases both act on bonds with substrates — some overlap is real.)

## 7. Compare with a Naive Baseline (Sequence Composition)

A common honest check: would you do as well using just amino acid composition (20-dim feature: % of each AA)? If yes, ESM-2 isn't pulling its weight.

In [ ]:
AA = 'ACDEFGHIKLMNPQRSTVWY'
def aa_composition(seq):
    counts = np.array([seq.count(a) for a in AA], dtype=np.float32)
    return counts / max(len(seq), 1)

X_aa = np.stack([aa_composition(s) for s in df['Sequence']])
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X_aa, y, test_size=0.2, random_state=42, stratify=y)

lr_aa = LogisticRegression(max_iter=2000, multi_class='multinomial').fit(Xa_tr, ya_tr)
print(f'AA composition only — test accuracy: {lr_aa.score(Xa_te, ya_te):.3f}')
print(f'ESM-2 + LR        — test accuracy: {accuracy_score(y_test, y_pred_lr):.3f}')
print(f'ESM-2 + MLP       — test accuracy: {accuracy_score(y_test, y_pred_mlp):.3f}')

## 8. Predicting on a New Sequence

Try the trained head on a sequence the model has never seen.

In [ ]:
# Carbonic anhydrase II (P00918), a well-known EC 4 lyase
test_seq = (
    'MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPLSVSYDQATSLRILNNGHAFNVEFDDSQDKAVL'
    'KGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQK'
)
with torch.no_grad():
    emb = embed_batch([test_seq])
    logits = head(torch.from_numpy(emb).float().to(device))
    probs = F.softmax(logits, dim=1).cpu().numpy()[0]

for i, p in enumerate(probs):
    print(f'  EC {i+1}: {p:.3f}')
print(f'Predicted: EC {probs.argmax()+1}  (true class: EC 4)')

## Reflection Questions

1. **Transfer learning intuition** — ESM-2 was never told what an enzyme is, yet a simple linear classifier on its embeddings achieves > 80% accuracy on a 4-way EC task. What does this tell you about what ESM-2 learned during pretraining?
2. **MLP vs LogReg** — Did the MLP beat logistic regression by much? On small data with strong features (like ESM-2 embeddings), linear heads are often hard to beat. Why?
3. **The biology** — What pairs of EC classes confuse the model? Look up an example confused enzyme — does the confusion make biological sense?
4. **Scaling up** — How would performance change if you used ESM-2 650M (the 80x larger model) instead? Try if you have time.

**Phase 3 B1 milestone:** Test accuracy ≥ 80% on EC classification, with a confusion-matrix interpretation that connects to enzyme biology.

# Phase 3 B1 — Enzyme Classifier 學習筆記

## 任務概述

用 ESM-2 的 protein embedding 訓練一個分類器，把酶分成四大類（EC1–EC4）。

```
蛋白質序列
  ↓ tokenizer（文字 → 數字）
  ↓ ESM-2 frozen（數字 → embedding 向量）  ← 不訓練
  ↓ 分類器（向量 → EC1/2/3/4）             ← 只訓練這裡
  ↓
預測結果
```

---

## 資料來源

- **UniProt SwissProt**（`reviewed:true`）：人工審核過的高品質資料庫
- EC 1–4 各下載 150 筆，`length:[50 TO 400]`
- 欄位：`Entry`（UniProt ID）、`Sequence`、`EC number`、`ec_top`（自行加的大類標籤）

### EC 四大類

| ec_top | 名稱 | 功能 |
|---|---|---|
| 1 | Oxidoreductase | 氧化還原（電子轉移） |
| 2 | Transferase | 把官能基從一個分子轉移到另一個 |
| 3 | Hydrolase | 水解反應 |
| 4 | Lyase | 不用水解就斷鍵或形成鍵 |

> 注意：有些蛋白質有多個 EC number（例如 `1.1.1.216; 1.1.1.300`），代表能催化多種反應。

---

## 核心概念

### Tokenizer vs Model

兩個獨立的物件，功能不同：

| 物件 | 作用 |
|---|---|
| `EsmTokenizer` | 把胺基酸序列（文字）轉成數字 ID |
| `EsmModel` | 把數字 ID 轉成 embedding 向量（神經網路） |

### Frozen vs 訓練

`for p in esm.parameters(): p.requires_grad = False`

這不是模型內建的設定，而是主動告訴 PyTorch「不要更新這些參數」。ESM-2 的權重（Meta 預訓練的蛋白質知識）在整個 notebook 裡完全不動，只訓練後面接的分類器。

---

## 實驗結果

### PCA 視覺化

四類 EC 在 PCA 2D 圖上大量重疊，無法用眼睛分開。

**原因**：EC 分類是按照催化的化學反應定義，ESM-2 學的是序列的演化和結構特徵，兩者在低維空間裡不完全對齊。但高維空間（320 維）裡可能還是可分的——這是分類器要驗證的事。

### 三種方法比較

| 方法 | 特徵 | 維度 | Test Accuracy |
|---|---|---|---|
| AA composition + LR | 胺基酸比例 | 20 | 0.385 |
| ESM-2 + LR | ESM-2 embedding | 320 | 0.726 |
| ESM-2 + MLP | ESM-2 embedding | 320 | 0.778 |

**關鍵結論**：最大的提升來自 ESM-2（+0.34），分類器從線性換非線性只是錦上添花（+0.04）。主角是 ESM-2。

### Confusion Matrix 觀察

- **EC1 Oxidoreductase**：precision 高（0.87）但 recall 低（0.67）→ 預測時很準，但漏掉很多真正的 EC1
- **EC2 Transferase**：最難分（precision 0.67, recall 0.62）→ 功能最多樣，序列差異大
- **EC3 Hydrolase**：最平衡（0.77 / 0.79）
- **EC4 Lyase**：recall 高（0.83）但 precision 低（0.65）→ 模型過度積極判成 EC4

**最常混淆的組合：EC2 ↔ EC4**

生物上合理：EC2 Transferase 和 EC4 Lyase 在反應機制上有重疊——某些 lyase 催化的消去反應和某些 transferase 在活性位點的化學環境很相似，演化上也有親緣關係。

---

## 反思問題

### Q1：ESM-2 從未被告知什麼是酶，為什麼還能分類？

ESM-2 預訓練只做「預測被遮住的胺基酸」（類似克漏字）。為了準確預測，它被迫學會哪些位置在演化上保守、哪些區域可能形成活性位點。這些**活性位點的化學環境**恰好和催化機制相關，所以 EC 分類的資訊隱含在 embedding 裡，即使模型從未看過 EC 標籤。

### Q2：MLP 只比 LR 多 0.04，為什麼？

ESM-2 的 embedding 品質很高，特徵已經整理得很好，LR 的線性邊界在這種情況下接近上限，MLP 能改善的空間本來就小。加上樣本數少（~480 筆），MLP 更容易 overfitting。

### Q3：Overfitting 的訊號

訓練時 loss 持續下降，但 test_acc 在某個點之後上下震盪不再進步——這就是 overfitting。本實驗最佳點約在 epoch 240–320，之後繼續跑沒有意義。解決方法是 **early stopping**。

### Q4：換成 ESM-2 650M 會更好嗎？

不一定。650M 的 embedding 維度從 320 升到 1280，特徵更細緻。但這個任務的瓶頸是**樣本數太少**，更大的模型反而可能加劇 overfitting。

---

## 技術細節備忘

```python
# 標籤從 [1,2,3,4] 轉成 [0,1,2,3]，因為 PyTorch 從 0 開始數
y = df['ec_top'].values - 1

# Learning rate
# 1e-3 = 0.001
# 1e-4 = 0.0001

# URL 查詢字串要用 urllib.parse.urlencode() 做正確編碼
# 不能手動拼字串，否則方括號等特殊字元會導致 HTTP 400
```